# 02 Generate Raw Tables

This notebook generates the **raw synthetic tables** for the Power BI Usage Intelligence project and saves them to `data/raw/`.

The goal of this notebook is to create a believable raw telemetry layer that can later be transformed into a clean semantic model with dimensions and facts.

## Tables created

- `reports`
- `users`
- `report_pages`
- `dates`
- `report_views`
- `report_page_views`
- `report_load_times`

## Why start with raw tables?

In a real analytics workflow, data usually arrives in a raw or source-aligned form before it is cleaned and modeled.  
This notebook mimics that pattern:

`raw tables -> clean semantic model -> features -> forecasting -> GenAI insights`


In [42]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)

PROJECT_ROOT = Path().resolve().parent
RAW_PATH = PROJECT_ROOT / "data" / "raw"

RAW_PATH.mkdir(parents=True, exist_ok=True)

print(f"Saving raw tables to: {RAW_PATH.resolve()}")

Saving raw tables to: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/raw


## Global simulation settings

These parameters control the size and time span of the synthetic data.

The numbers are intentionally modest so the notebook stays easy to run and inspect while still producing realistic-looking variation across reports, users, and dates.


In [43]:
N_REPORTS = 30
N_USERS = 200
START_DATE = "2025-01-01"
END_DATE = "2026-03-31"

dates_range = pd.date_range(START_DATE, END_DATE, freq="D")

print(f"Reports: {N_REPORTS}")
print(f"Users: {N_USERS}")
print(f"Date range: {START_DATE} to {END_DATE} ({len(dates_range)} days)")

Reports: 30
Users: 200
Date range: 2025-01-01 to 2026-03-31 (455 days)


## Create lookup tables

These tables provide the descriptive metadata that later becomes the backbone of the semantic model.


### `reports`

This table stores report-level metadata.

Each row represents one report.


In [44]:
report_types = ["Report", "Paginated", "Dashboard"]
workspace_ids = [f"WS_{i:03d}" for i in range(1, 6)]

reports = pd.DataFrame({
    "report_id": [f"R_{i:03d}" for i in range(1, N_REPORTS + 1)],
    "report_name": [f"Report_{i:03d}" for i in range(1, N_REPORTS + 1)],
    "workspace_id": np.random.choice(workspace_ids, N_REPORTS),
    "report_type": np.random.choice(report_types, N_REPORTS, p=[0.7, 0.2, 0.1]),
    "is_usage_metrics_report": np.random.choice([True, False], N_REPORTS, p=[0.1, 0.9])
})

reports.head()

,report_id,report_name,workspace_id,report_type,is_usage_metrics_report
0,R_001,Report_001,WS_004,Report,False
1,R_002,Report_002,WS_005,Report,False
2,R_003,Report_003,WS_003,Paginated,False
3,R_004,Report_004,WS_005,Report,True
4,R_005,Report_005,WS_005,Report,False


### `users`

This table stores user identifiers.

Each row represents one user.


In [45]:
users = pd.DataFrame({
    "user_key": [f"UK_{i:04d}" for i in range(1, N_USERS + 1)],
    "user_id": [f"user{i:03d}@masegoinc.com" for i in range(1, N_USERS + 1)],
    "unique_user": [f"User {i:03d}" for i in range(1, N_USERS + 1)]
})

users.head()

,user_key,user_id,unique_user
0,UK_0001,user001@masegoinc.com,User 001
1,UK_0002,user002@masegoinc.com,User 002
2,UK_0003,user003@masegoinc.com,User 003
3,UK_0004,user004@masegoinc.com,User 004
4,UK_0005,user005@masegoinc.com,User 005


### `report_pages`

This table stores page-level metadata for each report.

Each row represents one page within one report.


In [46]:
page_rows = []

for _, row in reports.iterrows():
    if row["report_type"] not in ["Report", "Dashboard"]:
        continue

    n_pages = np.random.randint(3, 9)

    for p in range(1, n_pages + 1):
        page_rows.append({
            "report_id": row["report_id"],
            "section_id": f"{row['report_id']}_P{p}",
            "section_name": f"Page {p}"
        })

report_pages = pd.DataFrame(page_rows)
report_pages.head()

,report_id,section_id,section_name
0,R_001,R_001_P1,Page 1
1,R_001,R_001_P2,Page 2
2,R_001,R_001_P3,Page 3
3,R_001,R_001_P4,Page 4
4,R_001,R_001_P5,Page 5


Validation check that only Reports and Dashboards have report pages. 

In [47]:
report_pages_check = report_pages.merge(
    reports[["report_id", "report_type"]],
    on="report_id",
    how="left"
)

report_pages_check["report_type"].value_counts()

report_type
Report       118
Dashboard     32
Name: count, dtype: int64

### `dates`

This table provides the calendar reference needed for time-based analysis.


In [48]:
dates = pd.DataFrame({"date": dates_range})
dates["day_of_week"] = dates["date"].dt.day_name()
dates["week_start_date"] = dates["date"] - pd.to_timedelta(dates["date"].dt.weekday, unit="D")
dates["month"] = dates["date"].dt.to_period("M").astype(str)
dates["is_weekend"] = dates["date"].dt.weekday >= 5

dates.head()

,date,day_of_week,week_start_date,month,is_weekend
0,2025-01-01,Wednesday,2024-12-30,2025-01,False
1,2025-01-02,Thursday,2024-12-30,2025-01,False
2,2025-01-03,Friday,2024-12-30,2025-01,False
3,2025-01-04,Saturday,2024-12-30,2025-01,True
4,2025-01-05,Sunday,2024-12-30,2025-01,True


## Hidden behavior drivers

Before generating usage events, we create a few hidden factors that make the synthetic data feel more realistic.

These are not final business tables. They are internal simulation drivers:

- **report popularity**: some reports are naturally more used than others
- **user activity**: some users are much more active than others
- **base load time**: some reports are naturally slower than others


### Why use a Gamma distribution for popularity and activity?

We use a Gamma distribution for `base_popularity` and `activity_score` because both represent **positive, right-skewed behaviour**.

That makes it a good fit for product and usage analytics because:

- values must be **positive**
- most reports and users should have **low to moderate activity**
- a smaller subset should behave like **popular reports** or **power users**

In real systems, usage is rarely evenly distributed. A few reports often drive a large share of total views, and a relatively small number of users often account for a disproportionate share of activity. The Gamma distribution helps approximate that long-tail pattern without making the synthetic data overly complex.

### Why not use a normal distribution?

A normal distribution would be less appropriate here because it:

- can generate negative values
- is symmetric
- creates too many average cases and too few realistic outliers

### Why not use a uniform distribution?

A uniform distribution would make all reports and users too similar, which would make the dataset feel artificial and reduce the usefulness of later feature engineering and forecasting.

### Practical interpretation in this notebook

The hidden simulation logic is:

- more popular reports are more likely to be viewed
- more active users are more likely to generate views
- the combination of the two influences view probability

That gives the synthetic data a more believable structure for later analysis.


In [49]:
report_popularity = pd.DataFrame({
    "report_id": reports["report_id"],
    "base_popularity": np.random.gamma(shape=2.0, scale=2.0, size=len(reports))
})

user_activity = pd.DataFrame({
    "user_key": users["user_key"],
    "activity_score": np.random.gamma(shape=2.0, scale=1.5, size=len(users))
})

report_performance = pd.DataFrame({
    "report_id": reports["report_id"],
    "base_load_time_ms": np.random.normal(loc=3500, scale=1200, size=len(reports)).clip(800, 9000)
})

report_popularity.head()

,report_id,base_popularity
0,R_001,2.797705
1,R_002,4.343027
2,R_003,1.651263
3,R_004,2.198394
4,R_005,6.299545


A quick look at the hidden drivers makes the long-tail behavior easier to see.


In [50]:
report_popularity["base_popularity"].describe()

count    30.000000
mean      3.884771
std       2.777800
min       0.428210
25%       2.239026
50%       3.378310
75%       4.321334
max      14.595996
Name: base_popularity, dtype: float64

In [51]:
user_activity["activity_score"].describe()

count    200.000000
mean       3.098097
std        2.138202
min        0.230270
25%        1.427395
50%        2.590689
75%        4.083656
max       10.194798
Name: activity_score, dtype: float64

## Create raw event-style tables

These are the main usage and performance tables that mimic raw telemetry.


### `report_views`

This is the primary behavioural table.

For this first version, the grain is:

**one row per `date x report x user` when at least one view occurs**

This keeps the data compact and easy to reason about while still supporting later transformation into a clean fact table.


In [52]:
view_rows = []

report_lookup = report_popularity.set_index("report_id")["base_popularity"].to_dict()
user_lookup = user_activity.set_index("user_key")["activity_score"].to_dict()

consumption_methods = ["Web", "Mobile"]
distribution_methods = ["Direct", "App","SharedLink", "Embedded"]

for date in dates_range:
    weekend_factor = 0.55 if date.weekday() >= 5 else 1.0

    for _, report in reports.iterrows():
        report_id = report["report_id"]
        popularity = report_lookup[report_id]

        for _, user in users.iterrows():
            user_key = user["user_key"]
            user_id = user["user_id"]
            activity = user_lookup[user_key]

            p_view = min(0.015 * popularity * activity * weekend_factor, 0.65)

            if np.random.rand() < p_view:
                view_count = np.random.choice([1, 2, 3, 4], p=[0.65, 0.2, 0.1, 0.05])

                view_rows.append({
                    "date": date,
                    "report_id": report_id,
                    "user_key": user_key,
                    "user_id": user_id,
                    "consumption_method": np.random.choice(consumption_methods, p=[0.85, 0.15]),
                    "distribution_method": np.random.choice(distribution_methods, p=[0.5, 0.3,0.1, 0.1]),
                    "user_agent": np.random.choice(["Chrome", "Edge", "Safari", "Firefox"], p=[0.45, 0.3, 0.15, 0.1]),
                    "view_count": view_count
                })

report_views = pd.DataFrame(view_rows)
report_views.head()

,date,report_id,user_key,user_id,consumption_method,distribution_method,user_agent,view_count
0,2025-01-01,R_001,UK_0002,user002@masegoinc.com,Web,Direct,Chrome,1
1,2025-01-01,R_001,UK_0004,user004@masegoinc.com,Web,App,Chrome,3
2,2025-01-01,R_001,UK_0012,user012@masegoinc.com,Mobile,App,Edge,2
3,2025-01-01,R_001,UK_0019,user019@masegoinc.com,Web,Direct,Edge,1
4,2025-01-01,R_001,UK_0023,user023@masegoinc.com,Web,Direct,Firefox,1


### `report_page_views`

This table is derived from report views so that page-level behavior is logically connected to report-level usage.

For each report-view row, we simulate one or more page views.


In [53]:
pages_by_report = report_pages.groupby("report_id")["section_id"].apply(list).to_dict()

page_view_rows = []

for _, row in report_views.iterrows():
    if row["report_id"] not in pages_by_report:
        continue

    possible_pages = pages_by_report[row["report_id"]]

    n_page_events = np.random.randint(1, min(len(possible_pages), row["view_count"] + 2) + 1)
    viewed_pages = np.random.choice(possible_pages, size=n_page_events, replace=False)

    for section_id in viewed_pages:
        ts = pd.Timestamp(row["date"]) + pd.Timedelta(minutes=np.random.randint(0, 1440))

        page_view_rows.append({
            "timestamp": ts,
            "date": row["date"],
            "report_id": row["report_id"],
            "section_id": section_id,
            "user_key": row["user_key"],
            "client": np.random.choice(["Browser", "MobileApp"], p=[0.85, 0.15]),
            "session_source": np.random.choice(["Direct", "App", "SharedLink"], p=[0.6, 0.3, 0.1]),
            "page_view_count": 1
        })

report_page_views = pd.DataFrame(page_view_rows)

### `report_load_times`

This table simulates performance telemetry for report loads.

We generate one load event for each report view row, using a report-specific baseline plus random noise.


In [54]:
load_rows = []

load_lookup = report_performance.set_index("report_id")["base_load_time_ms"].to_dict()

for _, row in report_views.iterrows():
    base_load = load_lookup[row["report_id"]]

    adjusted_load = np.random.normal(loc=base_load, scale=500)
    adjusted_load = max(500, adjusted_load)

    ts = pd.Timestamp(row["date"]) + pd.Timedelta(minutes=np.random.randint(0, 1440))

    load_rows.append({
        "timestamp": ts,
        "date": row["date"],
        "report_id": row["report_id"],
        "user_key": row["user_key"],
        "user_id": row["user_id"],
        "browser": row["user_agent"],
        "client": "Browser" if row["consumption_method"] == "Web" else "MobileApp",
        "country": np.random.choice(["Canada", "UK", "South Africa"], p=[0.7, 0.2, 0.1]),
        "load_time_ms": round(adjusted_load, 0)
    })

report_load_times = pd.DataFrame(load_rows)
report_load_times.head()

,timestamp,date,report_id,user_key,user_id,browser,client,country,load_time_ms
0,2025-01-01 13:25:00,2025-01-01,R_001,UK_0002,user002@masegoinc.com,Chrome,Browser,Canada,3937.0
1,2025-01-01 13:05:00,2025-01-01,R_001,UK_0004,user004@masegoinc.com,Chrome,Browser,UK,4902.0
2,2025-01-01 22:13:00,2025-01-01,R_001,UK_0012,user012@masegoinc.com,Edge,MobileApp,Canada,4087.0
3,2025-01-01 03:49:00,2025-01-01,R_001,UK_0019,user019@masegoinc.com,Edge,Browser,South Africa,4040.0
4,2025-01-01 17:03:00,2025-01-01,R_001,UK_0023,user023@masegoinc.com,Firefox,Browser,Canada,4460.0


## Basic sanity checks

These are simple checks to make sure the generated raw tables are coherent before saving them.


In [55]:
print("reports:", reports.shape)
print("users:", users.shape)
print("report_pages:", report_pages.shape)
print("dates:", dates.shape)
print("report_views:", report_views.shape)
print("report_page_views:", report_page_views.shape)
print("report_load_times:", report_load_times.shape)

reports: (30, 5)
users: (200, 3)
report_pages: (150, 3)
dates: (455, 5)
report_views: (411006, 8)
report_page_views: (879149, 8)
report_load_times: (411006, 9)


In [56]:
print("All report_ids in report_views exist in reports:",
      report_views["report_id"].isin(reports["report_id"]).all())

print("All user_keys in report_views exist in users:",
      report_views["user_key"].isin(users["user_key"]).all())

print("All section_ids in report_page_views exist in report_pages:",
      report_page_views["section_id"].isin(report_pages["section_id"]).all())

print("All report_ids in report_load_times exist in reports:",
      report_load_times["report_id"].isin(reports["report_id"]).all())

All report_ids in report_views exist in reports: True
All user_keys in report_views exist in users: True
All section_ids in report_page_views exist in report_pages: True
All report_ids in report_load_times exist in reports: True


## Save raw tables

Save the generated tables to `data/raw/` in parquet format.

Parquet is a good choice because it is compact, fast, and common in analytics workflows.


In [57]:
reports.to_csv(RAW_PATH / "reports.csv", index=False)
users.to_csv(RAW_PATH / "users.csv", index=False)
report_pages.to_csv(RAW_PATH / "report_pages.csv", index=False)
dates.to_csv(RAW_PATH / "dates.csv", index=False)
report_views.to_csv(RAW_PATH / "report_views.csv", index=False)
report_page_views.to_csv(RAW_PATH / "report_page_views.csv", index=False)
report_load_times.to_csv(RAW_PATH / "report_load_times.csv", index=False)

## Next step

After this notebook, the next step is to build the clean semantic model:

- `dim_date`
- `dim_user`
- `dim_report`
- `dim_page`
- `fact_report_views`
- `fact_page_views`
- `fact_report_loads`

Those cleaned tables will be saved separately from the raw layer and will become the main input for feature engineering and forecasting.
